# SE-ResNeXt-50 Sampler and Native-CAM Ablation

The completed SE-ResNeXt final-native-CAM run is currently the strongest localization
configuration. Its main weakness is Grade 0/1 confusion and low Grade 1 precision,
while the rejected multiscale+EMA run increased Grade 1 recall by over-predicting
Grade 1 and degraded QWK, macro F1, joint energy, and border energy.

This controlled experiment changes only sampler strength:

1. full inverse-frequency sampling (`count^-1`), the current baseline;
2. square-root inverse-frequency sampling (`count^-0.5`), a less aggressive balance;
3. no weighted sampler, ordinary shuffled training.

The final 12x12 native-CAM architecture, split, transforms, CE loss, schedule,
learning rates, and validation selection score remain fixed. The test split is not
read; the selected candidate must be evaluated later on the locked holdout.


## Run Instructions

Run every cell on a fresh runtime. Each variant receives its own timestamped
checkpoint directory. The notebook audits 50 validation cases per grade for joint
energy, border energy, lower-tibia energy, peak location, and 4x4 occlusion
faithfulness.

No sampler variant is automatically promoted. A candidate is eligible only when its
validation selection score is at least the current baseline (`0.7003`) and its CAM
joint/border metrics do not regress from the baseline (`0.8707` / `0.0749`).


In [1]:
import csv
import gc
import hashlib
import json
import random
import subprocess
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
import tqdm
import timm
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    cohen_kappa_score,
    precision_recall_fscore_support,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab; Drive mount skipped.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

SEED = 42
BATCH_SIZE = 48
NUM_WORKERS = 4
STAGE_EPOCHS = (5, 15, 10)
CAM_AUDIT_PER_GRADE = 50
OCCLUSION_GRID = 4
BASELINE_VALIDATION_SELECTION = 0.7002724504801549
BASELINE_CAM_JOINT = 0.8706657083596738
BASELINE_CAM_BORDER = 0.07487359307599288

SAMPLER_VARIANTS = [
    {"name": "full_inverse", "sampler_power": 1.0},
    {"name": "sqrt_inverse", "sampler_power": 0.5},
    {"name": "no_sampler", "sampler_power": None},
]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

dataset_zip = Path("/content/drive/MyDrive/Datasets/kaggle_knee_osteoarthritis.zip")
if dataset_zip.exists():
    subprocess.run(["unzip", "-q", "-o", str(dataset_zip), "-d", "/content/Datasets"], check=True)
DATASET_ROOT = Path("/content/Datasets/kaggle_knee_osteoarthritis")
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S_%f_UTC")
RUN_ROOT = (
    Path("/content/drive/MyDrive/Models/seresnext_sampler_ablations")
    / f"{RUN_TIMESTAMP}_full_vs_sqrt_vs_none"
)
RUN_ROOT.mkdir(parents=True, exist_ok=False)
print(f"Ablation root: {RUN_ROOT}")


Mounted at /content/drive
Device: cuda
GPU: Tesla T4
Ablation root: /content/drive/MyDrive/Models/seresnext_sampler_ablations/2026-07-23_15-13-05_616211_UTC_full_vs_sqrt_vs_none


## Data and Fixed Transforms

In [2]:
class SquarePad:
    def __call__(self, image):
        height, width = image.shape[:2]
        side = max(height, width)
        top = (side - height) // 2
        bottom = side - height - top
        left = (side - width) // 2
        right = side - width - left
        return cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT, value=[0, 0, 0])


class CLAHE:
    def __call__(self, image):
        lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
        lightness, channel_a, channel_b = cv2.split(lab)
        lightness = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lightness)
        return cv2.cvtColor(cv2.merge((lightness, channel_a, channel_b)), cv2.COLOR_LAB2RGB)


train_transform = transforms.Compose([
    SquarePad(), CLAHE(), transforms.ToPILImage(), transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.08, contrast=0.08), transforms.Resize((400, 400)),
    transforms.RandomCrop(384), transforms.ToTensor(),
    transforms.RandomErasing(p=0.10, scale=(0.02, 0.05), ratio=(0.5, 2.0), value=0),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
evaluation_transform = transforms.Compose([
    SquarePad(), CLAHE(), transforms.ToPILImage(), transforms.Resize((400, 400)),
    transforms.CenterCrop(384), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def file_digest(path):
    digest = hashlib.sha256()
    with open(path, "rb") as image_file:
        for chunk in iter(lambda: image_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


class KneeDataset(Dataset):
    def __init__(self, split, transform, excluded_hashes=None):
        self.transform = transform
        self.paths, self.labels = [], []
        excluded_hashes = excluded_hashes or set()
        retained_hashes = set()
        for grade in range(5):
            grade_dir = DATASET_ROOT / split / str(grade)
            if not grade_dir.is_dir():
                raise FileNotFoundError(f"Missing dataset directory: {grade_dir}")
            for path in sorted(grade_dir.iterdir()):
                if path.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
                    continue
                digest = file_digest(path)
                if digest in excluded_hashes or digest in retained_hashes:
                    continue
                retained_hashes.add(digest)
                self.paths.append(str(path))
                self.labels.append(grade)
        self.image_hashes = retained_hashes
        print(f"{split}: {len(self.paths)} unique images")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        image = cv2.imread(path)
        if image is None:
            raise IOError(f"Could not read image: {path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if Path(path).stem.upper().endswith("R"):
            image = np.ascontiguousarray(image[:, ::-1])
        return self.transform(image), self.labels[index], path


train_dataset = KneeDataset("train", train_transform)
validation_dataset = KneeDataset("val", evaluation_transform, train_dataset.image_hashes)
class_counts = Counter(train_dataset.labels)
print("Class counts:", dict(sorted(class_counts.items())))


train: 5778 unique images
val: 826 unique images
Class counts: {0: 2286, 1: 1046, 2: 1516, 3: 757, 4: 173}


## Fixed SE-ResNeXt Native-CAM Model and Training

In [3]:
class SEResNeXtNativeCAM(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            "seresnext50_32x4d", pretrained=True, features_only=True, out_indices=(4,)
        )
        self.class_conv = nn.Conv2d(self.backbone.feature_info.channels()[0], 5, kernel_size=1)

    def head_parameters(self):
        return self.class_conv.parameters()

    def class_maps(self, images):
        return self.class_conv(self.backbone(images)[0])

    def forward(self, images):
        return self.class_maps(images).mean(dim=(2, 3))

    def native_cam(self, images, class_index):
        with torch.no_grad():
            maps = self.class_maps(images)
            logits = maps.mean(dim=(2, 3))
            cam = F.relu(maps[:, class_index : class_index + 1])
            cam = F.interpolate(cam, size=images.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
            cam = cam / cam.max().clamp_min(1e-8)
        return cam, logits

    def freeze_backbone(self):
        for parameter in self.backbone.parameters():
            parameter.requires_grad = False
        for parameter in self.class_conv.parameters():
            parameter.requires_grad = True

    def unfreeze_final_stages(self):
        for parameter in self.parameters():
            parameter.requires_grad = False
        for name, parameter in self.backbone.named_parameters():
            if any(token in name for token in ("layer3", "layer4", "stages.2", "stages.3")):
                parameter.requires_grad = True
        for parameter in self.class_conv.parameters():
            parameter.requires_grad = True

    def unfreeze_all(self):
        for parameter in self.parameters():
            parameter.requires_grad = True


def keep_frozen_batch_norm_eval(model):
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d) and not any(parameter.requires_grad for parameter in module.parameters()):
            module.eval()


def make_loaders(variant):
    generator = torch.Generator().manual_seed(SEED)
    if variant["sampler_power"] is None:
        sampler = None
        shuffle = True
    else:
        weights = [class_counts[label] ** (-variant["sampler_power"]) for label in train_dataset.labels]
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True, generator=generator)
        shuffle = False
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, sampler=sampler, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, generator=generator,
    )
    validation_loader = DataLoader(
        validation_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True,
    )
    return train_loader, validation_loader


def calculate_metrics(labels, predictions, probabilities):
    labels, predictions, probabilities = np.asarray(labels), np.asarray(predictions), np.asarray(probabilities)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, labels=[0, 1, 2, 3, 4], average="macro", zero_division=0)
    one_hot = np.eye(5)[labels]
    metrics = {
        "accuracy": float(accuracy_score(labels, predictions)),
        "qwk": float(cohen_kappa_score(labels, predictions, weights="quadratic")),
        "macro_precision": float(precision), "macro_recall": float(recall), "macro_f1": float(f1),
        "grade1_recall": float(recall_score(labels == 1, predictions == 1, zero_division=0)),
        "macro_ap": float(average_precision_score(one_hot, probabilities, average="macro")),
        "macro_auc": float(roc_auc_score(one_hot, probabilities, average="macro")),
    }
    metrics["selection_score"] = float(
        0.40 * metrics["qwk"] + 0.20 * metrics["macro_f1"] + 0.10 * metrics["macro_recall"]
        + 0.10 * metrics["grade1_recall"] + 0.15 * metrics["macro_ap"] + 0.05 * metrics["macro_auc"]
    )
    return metrics


def evaluate(model, loader):
    model.eval()
    labels_all, predictions_all, probabilities_all = [], [], []
    with torch.no_grad():
        for images, labels, _ in loader:
            logits = model(images.to(device, non_blocking=True))
            probabilities = F.softmax(logits.float(), dim=1)
            labels_all.extend(labels.numpy())
            predictions_all.extend(probabilities.argmax(1).cpu().numpy())
            probabilities_all.extend(probabilities.cpu().numpy())
    return calculate_metrics(labels_all, predictions_all, probabilities_all)


def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    keep_frozen_batch_norm_eval(model)
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    for images, labels, _ in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.type, enabled=device.type == "cuda"):
            loss = F.cross_entropy(model(images), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        running_loss += loss.item() * labels.size(0)
    return running_loss / len(loader.dataset)


def load_checkpoint(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


## Train the Three Sampler Variants

In [4]:
def train_variant(variant):
    started = time.time()
    variant_dir = RUN_ROOT / variant["name"]
    variant_dir.mkdir(parents=True, exist_ok=False)
    train_loader, validation_loader = make_loaders(variant)
    model = SEResNeXtNativeCAM().to(device)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    history, best_score, best_stage2_score, global_epoch = [], -float("inf"), -float("inf"), 0
    best_path = variant_dir / "best_model.pth"
    stage2_path = variant_dir / "stage2_best_model.pth"
    last_path = variant_dir / "last_model.pth"

    for stage_name, stage_epochs in zip(("warmup", "coarse", "finetune"), STAGE_EPOCHS):
        if stage_name == "warmup":
            model.freeze_backbone()
            optimizer = optim.AdamW(model.head_parameters(), lr=3e-4, weight_decay=1e-4)
            scheduler = None
        elif stage_name == "coarse":
            model.unfreeze_final_stages()
            optimizer = optim.AdamW(
                [
                    {"params": [parameter for parameter in model.backbone.parameters() if parameter.requires_grad], "lr": 3e-5},
                    {"params": model.head_parameters(), "lr": 3e-4},
                ], weight_decay=1e-4,
            )
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=stage_epochs, eta_min=1e-7)
        else:
            model.load_state_dict(load_checkpoint(stage2_path)["model_state_dict"])
            model.unfreeze_all()
            optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-3)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=stage_epochs, eta_min=1e-7)

        for stage_epoch in range(stage_epochs):
            global_epoch += 1
            train_loss = train_one_epoch(model, train_loader, optimizer, scaler)
            validation_metrics = evaluate(model, validation_loader)
            history.append({"epoch": global_epoch, "stage": stage_name, "train_loss": train_loss, **validation_metrics})
            print(f"{variant['name']} | epoch {global_epoch:02d} | {stage_name} | QWK={validation_metrics['qwk']:.4f} | F1={validation_metrics['macro_f1']:.4f} | G1R={validation_metrics['grade1_recall']:.4f} | selection={validation_metrics['selection_score']:.4f}")
            if scheduler is not None:
                scheduler.step()
            payload = {
                "model_state_dict": model.state_dict(), "epoch": global_epoch, "stage": stage_name,
                "architecture": "final_native_cam_ce", "model_name": "seresnext50_32x4d", "loss_type": "ce",
                "validation_metrics": validation_metrics, "history": history, "run_timestamp": RUN_TIMESTAMP,
                "experimental_config": {
                    "sampler": variant["name"], "sampler_power": variant["sampler_power"],
                    "batch_size": BATCH_SIZE, "stage_epochs": list(STAGE_EPOCHS),
                    "native_cam_source_resolution": 12, "ema": False,
                },
            }
            if stage_name == "coarse" and validation_metrics["selection_score"] > best_stage2_score:
                best_stage2_score = validation_metrics["selection_score"]
                torch.save(payload, stage2_path)
            if stage_name == "finetune" and validation_metrics["selection_score"] > best_score:
                best_score = validation_metrics["selection_score"]
                torch.save(payload, best_path)
            torch.save(payload, last_path)

    with open(variant_dir / "epoch_metrics.csv", "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=history[0].keys())
        writer.writeheader(); writer.writerows(history)
    selected = load_checkpoint(best_path)
    manifest = {
        "variant": variant["name"], "sampler_power": variant["sampler_power"],
        "run_timestamp": RUN_TIMESTAMP, "best_epoch": selected["epoch"],
        "validation_metrics": selected["validation_metrics"],
        "elapsed_minutes": (time.time() - started) / 60.0, "checkpoint": str(best_path),
    }
    with open(variant_dir / "run_manifest.json", "w") as handle:
        json.dump(manifest, handle, indent=2)
    del model, train_loader, validation_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return manifest, variant_dir


trained = {}
for variant in SAMPLER_VARIANTS:
    manifest, variant_dir = train_variant(variant)
    trained[variant["name"]] = {"manifest": manifest, "directory": variant_dir}


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model.safetensors: reconstructing file:   0%|          |  0.00B /  111MB            

model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


full_inverse | epoch 01 | warmup | QWK=0.4942 | F1=0.3413 | G1R=0.1503 | selection=0.4189
full_inverse | epoch 02 | warmup | QWK=0.4831 | F1=0.2974 | G1R=0.0131 | selection=0.3931
full_inverse | epoch 03 | warmup | QWK=0.5583 | F1=0.3464 | G1R=0.4118 | selection=0.4794
full_inverse | epoch 04 | warmup | QWK=0.5502 | F1=0.3368 | G1R=0.1438 | selection=0.4492
full_inverse | epoch 05 | warmup | QWK=0.5731 | F1=0.3732 | G1R=0.3399 | selection=0.4900
full_inverse | epoch 06 | coarse | QWK=0.6610 | F1=0.4495 | G1R=0.0261 | selection=0.5323
full_inverse | epoch 07 | coarse | QWK=0.7150 | F1=0.5176 | G1R=0.0654 | selection=0.5884
full_inverse | epoch 08 | coarse | QWK=0.7913 | F1=0.6037 | G1R=0.2549 | selection=0.6641
full_inverse | epoch 09 | coarse | QWK=0.7721 | F1=0.5923 | G1R=0.2222 | selection=0.6517
full_inverse | epoch 10 | coarse | QWK=0.7486 | F1=0.5867 | G1R=0.3725 | selection=0.6586
full_inverse | epoch 11 | coarse | QWK=0.7797 | F1=0.6079 | G1R=0.4118 | selection=0.6832
full_inver

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


sqrt_inverse | epoch 01 | warmup | QWK=0.2777 | F1=0.2108 | G1R=0.0000 | selection=0.2760
sqrt_inverse | epoch 02 | warmup | QWK=0.3051 | F1=0.2141 | G1R=0.0000 | selection=0.2912
sqrt_inverse | epoch 03 | warmup | QWK=0.4051 | F1=0.2810 | G1R=0.0000 | selection=0.3509
sqrt_inverse | epoch 04 | warmup | QWK=0.4665 | F1=0.3538 | G1R=0.0000 | selection=0.3986
sqrt_inverse | epoch 05 | warmup | QWK=0.4988 | F1=0.3457 | G1R=0.0000 | selection=0.4105
sqrt_inverse | epoch 06 | coarse | QWK=0.6490 | F1=0.4327 | G1R=0.0000 | selection=0.5133
sqrt_inverse | epoch 07 | coarse | QWK=0.7340 | F1=0.5331 | G1R=0.0523 | selection=0.5965
sqrt_inverse | epoch 08 | coarse | QWK=0.7609 | F1=0.5794 | G1R=0.0523 | selection=0.6261
sqrt_inverse | epoch 09 | coarse | QWK=0.7679 | F1=0.5822 | G1R=0.0588 | selection=0.6304
sqrt_inverse | epoch 10 | coarse | QWK=0.7537 | F1=0.5884 | G1R=0.1961 | selection=0.6413
sqrt_inverse | epoch 11 | coarse | QWK=0.7929 | F1=0.6089 | G1R=0.3529 | selection=0.6797
sqrt_inver

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


no_sampler | epoch 01 | warmup | QWK=0.0738 | F1=0.1277 | G1R=0.0000 | selection=0.1691
no_sampler | epoch 02 | warmup | QWK=0.2458 | F1=0.1733 | G1R=0.0000 | selection=0.2565
no_sampler | epoch 03 | warmup | QWK=0.3875 | F1=0.2307 | G1R=0.0000 | selection=0.3312
no_sampler | epoch 04 | warmup | QWK=0.3234 | F1=0.2036 | G1R=0.0000 | selection=0.2994
no_sampler | epoch 05 | warmup | QWK=0.3609 | F1=0.2084 | G1R=0.0000 | selection=0.3166
no_sampler | epoch 06 | coarse | QWK=0.6179 | F1=0.3085 | G1R=0.0000 | selection=0.4599
no_sampler | epoch 07 | coarse | QWK=0.7248 | F1=0.3912 | G1R=0.0000 | selection=0.5408
no_sampler | epoch 08 | coarse | QWK=0.7558 | F1=0.4996 | G1R=0.0392 | selection=0.5940
no_sampler | epoch 09 | coarse | QWK=0.7744 | F1=0.6013 | G1R=0.1242 | selection=0.6421
no_sampler | epoch 10 | coarse | QWK=0.7863 | F1=0.6124 | G1R=0.1111 | selection=0.6526
no_sampler | epoch 11 | coarse | QWK=0.7842 | F1=0.6315 | G1R=0.1961 | selection=0.6664
no_sampler | epoch 12 | coarse |

## 50-Per-Grade Native-CAM and Occlusion Audit

In [5]:
def cam_geometry(cam):
    height, width = cam.shape
    joint = np.zeros((height, width), dtype=bool)
    joint[int(0.28 * height):int(0.72 * height), int(0.06 * width):int(0.94 * width)] = True
    border = np.ones((height, width), dtype=bool)
    border[int(0.08 * height):int(0.92 * height), int(0.08 * width):int(0.92 * width)] = False
    lower_tibia = np.zeros((height, width), dtype=bool)
    lower_tibia[int(0.72 * height):, :] = True
    total = float(cam.sum()) + 1e-8
    peak = np.unravel_index(np.argmax(cam), cam.shape)
    return {
        "joint_energy": float(cam[joint].sum()) / total,
        "border_energy": float(cam[border].sum()) / total,
        "lower_tibia_energy": float(cam[lower_tibia].sum()) / total,
        "peak_inside_joint": int(joint[peak]),
    }


def occlusion_metrics(model, image, target_class, cam):
    height, width = image.shape[-2:]
    patch_height, patch_width = height // OCCLUSION_GRID, width // OCCLUSION_GRID
    variants, energies = [], []
    for row in range(OCCLUSION_GRID):
        for column in range(OCCLUSION_GRID):
            y0, y1 = row * patch_height, height if row == OCCLUSION_GRID - 1 else (row + 1) * patch_height
            x0, x1 = column * patch_width, width if column == OCCLUSION_GRID - 1 else (column + 1) * patch_width
            variant = image.clone(); variant[:, y0:y1, x0:x1] = 0.0
            variants.append(variant); energies.append(float(cam[y0:y1, x0:x1].sum()))
    with torch.no_grad():
        baseline = F.softmax(model(image[None]), dim=1)[0, target_class]
        probabilities = F.softmax(model(torch.stack(variants).to(device)), dim=1)[:, target_class]
    drops, energies = (baseline - probabilities).cpu().numpy(), np.asarray(energies)
    correlation = 0.0 if np.std(energies) < 1e-8 or np.std(drops) < 1e-8 else float(np.corrcoef(energies, drops)[0, 1])
    top = np.argsort(energies)[-max(1, len(energies) // 4):]
    return {"occlusion_correlation": correlation, "top_cam_confidence_drop": float(np.mean(drops[top]))}


rng = np.random.default_rng(SEED)
labels_array = np.asarray(validation_dataset.labels)
audit_indices = []
for grade in range(5):
    grade_indices = np.flatnonzero(labels_array == grade); rng.shuffle(grade_indices)
    audit_indices.extend(grade_indices[:min(CAM_AUDIT_PER_GRADE, len(grade_indices))].tolist())

cam_summaries = []
for variant in SAMPLER_VARIANTS:
    entry = trained[variant["name"]]
    model = SEResNeXtNativeCAM().to(device)
    model.load_state_dict(load_checkpoint(entry["directory"] / "best_model.pth")["model_state_dict"])
    model.eval(); rows = []
    for index in tqdm.tqdm(audit_indices, desc=f"CAM audit {variant['name']}"):
        tensor, true_grade, path = validation_dataset[index]
        image = tensor.to(device)
        with torch.no_grad():
            logits = model(image[None]); predicted_grade = int(logits.argmax(1).item())
            cam, _ = model.native_cam(image[None], predicted_grade)
            cam = cam.cpu().numpy()
        rows.append({
            "path": path, "true_grade": int(true_grade), "predicted_grade": predicted_grade,
            **cam_geometry(cam), **occlusion_metrics(model, image, predicted_grade, cam),
        })
    with open(entry["directory"] / "native_cam_audit.csv", "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=rows[0].keys()); writer.writeheader(); writer.writerows(rows)
    summary = {
        "variant": variant["name"], "audited_cases": len(rows), "source_resolution": 12,
        "joint_energy": float(np.mean([row["joint_energy"] for row in rows])),
        "border_energy": float(np.mean([row["border_energy"] for row in rows])),
        "lower_tibia_energy": float(np.mean([row["lower_tibia_energy"] for row in rows])),
        "peak_inside_joint_rate": float(np.mean([row["peak_inside_joint"] for row in rows])),
        "occlusion_correlation": float(np.mean([row["occlusion_correlation"] for row in rows])),
        "top_cam_confidence_drop": float(np.mean([row["top_cam_confidence_drop"] for row in rows])),
    }
    with open(entry["directory"] / "native_cam_summary.json", "w") as handle:
        json.dump(summary, handle, indent=2)
    entry["manifest"]["native_cam_summary"] = summary
    with open(entry["directory"] / "run_manifest.json", "w") as handle:
        json.dump(entry["manifest"], handle, indent=2)
    cam_summaries.append(summary)
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


CAM audit no_sampler: 100%|██████████| 227/227 [01:05<00:00,  3.47it/s]


## Comparison and Fixed Promotion Gate

In [6]:
import pandas as pd

metric_rows = []
for name, entry in trained.items():
    metric_rows.append({"variant": name, "best_epoch": entry["manifest"]["best_epoch"], **entry["manifest"]["validation_metrics"]})
comparison = pd.DataFrame(metric_rows).merge(pd.DataFrame(cam_summaries), on="variant", how="inner")
comparison["predictive_gate"] = comparison["selection_score"] >= BASELINE_VALIDATION_SELECTION
comparison["localization_gate"] = (
    (comparison["joint_energy"] >= BASELINE_CAM_JOINT)
    & (comparison["border_energy"] <= BASELINE_CAM_BORDER)
)
comparison["eligible_for_followup_holdout"] = comparison["predictive_gate"] & comparison["localization_gate"]
comparison = comparison.sort_values("selection_score", ascending=False)
comparison.to_csv(RUN_ROOT / "sampler_comparison.csv", index=False)
comparison.to_json(RUN_ROOT / "sampler_comparison.json", orient="records", indent=2)
display(comparison[[
    "variant", "best_epoch", "selection_score", "qwk", "macro_f1", "grade1_recall",
    "joint_energy", "border_energy", "peak_inside_joint_rate", "occlusion_correlation",
    "predictive_gate", "localization_gate", "eligible_for_followup_holdout",
]].round(4))

winner = comparison.iloc[0]
(RUN_ROOT / "VALIDATION_WINNER.txt").write_text(
    f"{winner['variant']}\nRun one locked holdout evaluation only if eligible_for_followup_holdout is True.\n"
)

report = f'''# SE-ResNeXt Sampler Ablation Report

| Field | Value |
| --- | --- |
| Exact UTC run timestamp | {RUN_TIMESTAMP} |
| Architecture | Final 12x12 native-CAM CE |
| Variants | Full inverse, square-root inverse, no sampler |
| Test split | Not read |
| CAM audit | Up to {CAM_AUDIT_PER_GRADE} cases per grade per variant |
| Validation winner | `{winner['variant']}` |

## Results

| Variant | Selection | QWK | Macro F1 | Grade 1 recall | Joint energy | Border energy | Peak inside | Occlusion correlation | Eligible |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | --- |
'''
for _, row in comparison.iterrows():
    report += (
        f"| {row['variant']} | {row['selection_score']:.4f} | {row['qwk']:.4f} | "
        f"{row['macro_f1']:.4f} | {row['grade1_recall']:.4f} | {row['joint_energy']:.4f} | "
        f"{row['border_energy']:.4f} | {row['peak_inside_joint_rate']:.4f} | "
        f"{row['occlusion_correlation']:.4f} | {bool(row['eligible_for_followup_holdout'])} |\n"
    )
report += (
    "\nThe test split was deliberately excluded. The selected validation variant is "
    "only a follow-up candidate; evaluate it once on the locked holdout after reviewing "
    "this report and its CAM gallery.\n"
)
(RUN_ROOT / "report.md").write_text(report, encoding="utf-8")
print(report)
print(f"All ablation artifacts: {RUN_ROOT}")


,variant,best_epoch,selection_score,qwk,macro_f1,grade1_recall,joint_energy,border_energy,peak_inside_joint_rate,occlusion_correlation,predictive_gate,localization_gate,eligible_for_followup_holdout
0,full_inverse,30,0.6984,0.7948,0.6446,0.3725,0.8421,0.0898,0.9956,0.5745,False,False,False
1,sqrt_inverse,30,0.6875,0.7895,0.6496,0.3072,0.8187,0.1048,0.9912,0.5841,False,False,False
2,no_sampler,27,0.6650,0.7737,0.6348,0.1961,0.8067,0.1117,0.9736,0.6001,False,False,False


# SE-ResNeXt Sampler Ablation Report

| Field | Value |
| --- | --- |
| Exact UTC run timestamp | 2026-07-23_15-13-05_616211_UTC |
| Architecture | Final 12x12 native-CAM CE |
| Variants | Full inverse, square-root inverse, no sampler |
| Test split | Not read |
| CAM audit | Up to 50 cases per grade per variant |
| Validation winner | `full_inverse` |

## Results

| Variant | Selection | QWK | Macro F1 | Grade 1 recall | Joint energy | Border energy | Peak inside | Occlusion correlation | Eligible |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | --- |
| full_inverse | 0.6984 | 0.7948 | 0.6446 | 0.3725 | 0.8421 | 0.0898 | 0.9956 | 0.5745 | False |
| sqrt_inverse | 0.6875 | 0.7895 | 0.6496 | 0.3072 | 0.8187 | 0.1048 | 0.9912 | 0.5841 | False |
| no_sampler | 0.6650 | 0.7737 | 0.6348 | 0.1961 | 0.8067 | 0.1117 | 0.9736 | 0.6001 | False |

The test split was deliberately excluded. The selected validation variant is only a follow-up candidate; evaluate it once on the locke

In [7]:
try:
    from google.colab import runtime
    print("All artifacts saved. Releasing the Colab runtime.")
    runtime.unassign()
except ImportError:
    print("Not running in Colab; runtime release skipped.")


All artifacts saved. Releasing the Colab runtime.
